# 01 · Limpieza de datos y modelo relacional

**Proyecto:** Salud mental en el sector tecnológico · De una encuesta larga a 7 preguntas
**Autor:** Hugo Nascimento · Proyecto final Data Analytics · IT Academy (Barcelona Activa)

**Datos:** he utilizado la encuesta *OSMI Mental Health in Tech* (Open Sourcing Mental Health) de 2014, 2016, 2017, 2018 y 2019. La he descargado de Kaggle en la versión SQLite que publicó Anthony Inthavong (licencia CC BY-SA 4.0). Son respuestas reales de personas que trabajan en el sector tecnológico.

**Lo que hago en este notebook:**
1. Reviso cómo está organizada la base de datos original y si los datos son de calidad.
2. Limpio los datos y explico cada decisión que tomo.
3. Construyo un modelo relacional de 7 tablas con claves primarias y foráneas.
4. Guardo el resultado en CSV (para Power BI), en SQLite (para Python) y en un script para crear la base en MySQL.

In [1]:
import re
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path.cwd().parent
RUTA_ORIGINAL = BASE / "01_datos" / "originales" / "mental_health.sqlite"
RUTA_PROCESADOS = BASE / "01_datos" / "procesados"
RUTA_AUXILIARES = BASE / "01_datos" / "auxiliares"
RUTA_SQLITE = RUTA_PROCESADOS / "salud_mental_tech.db"
RUTA_MYSQL = BASE / "03_sql" / "01_crear_base_datos.sql"
RUTA_PROCESADOS.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 90)

## 1. Cómo es la base de datos original

La base original tiene 3 tablas:

| Tabla | Qué tiene |
|---|---|
| `Survey` | Una fila por cada encuesta anual (5) |
| `Question` | Una fila por pregunta (105) |
| `Answer` | Una fila por respuesta, con el texto, la encuesta, la persona y la pregunta |

In [2]:
with sqlite3.connect(RUTA_ORIGINAL) as con:
    esquema = pd.read_sql("SELECT name, sql FROM sqlite_master WHERE type = 'table'", con)
    survey = pd.read_sql("SELECT * FROM Survey", con)
    question = pd.read_sql("SELECT * FROM Question", con)
    answer = pd.read_sql("SELECT * FROM Answer", con)

for _, fila in esquema.iterrows():
    print(fila["sql"], "\n")
print(f"Survey: {len(survey)} filas | Question: {len(question)} filas | Answer: {len(answer):,} filas")

CREATE TABLE "Answer" (
	"AnswerText" VARCHAR(10000) NULL,
	"SurveyID" INTEGER NULL,
	"UserID" INTEGER NULL,
	"QuestionID" INTEGER NULL
) 

CREATE TABLE "Question" (
	"questiontext" VARCHAR(1000) NULL,
	"questionid" INTEGER NULL
) 

CREATE TABLE "Survey" (
	"SurveyID" INTEGER NOT NULL,
	"Description" VARCHAR(255) NULL,
	PRIMARY KEY ("SurveyID")
) 

Survey: 5 filas | Question: 105 filas | Answer: 236,898 filas


Solo la tabla `Survey` tiene clave primaria. `Question` y `Answer` no tienen claves primarias ni foráneas, así que la base no asegura que los datos estén bien relacionados (por ejemplo, podría haber respuestas a preguntas que no existen). En mi modelo las voy a definir yo.

## 2. Control de calidad

In [3]:
def resumen_calidad(answer, question, survey):
    """Devuelve un diccionario con los controles básicos de calidad de la tabla de respuestas."""
    return {
        "respuestas": len(answer),
        "personas distintas": answer["UserID"].nunique(),
        "personas en más de una encuesta": int((answer.groupby("UserID")["SurveyID"].nunique() > 1).sum()),
        "respuestas a preguntas inexistentes": int((~answer["QuestionID"].isin(question["questionid"])).sum()),
        "respuestas de encuestas inexistentes": int((~answer["SurveyID"].isin(survey["SurveyID"])).sum()),
        "respuestas con código -1": int((answer["AnswerText"] == "-1").sum()),
        "respuestas vacías": int((answer["AnswerText"].str.strip() == "").sum()),
        "filas repetidas (persona + pregunta)": int(answer.duplicated(["UserID", "QuestionID"]).sum()),
        "IDs de pregunta duplicados": int(question["questionid"].duplicated().sum()),
    }

pd.Series(resumen_calidad(answer, question, survey), name="valor").to_frame()

,valor
respuestas,236898
personas distintas,4218
personas en más de una encuesta,0
respuestas a preguntas inexistentes,0
respuestas de encuestas inexistentes,0
respuestas con código -1,56598
respuestas vacías,25
filas repetidas (persona + pregunta),2258
IDs de pregunta duplicados,0


In [4]:
personas_por_anio = answer.groupby("SurveyID").agg(personas=("UserID", "nunique"),
                                                  respuestas=("AnswerText", "size"),
                                                  preguntas=("QuestionID", "nunique"))
personas_por_anio

,personas,respuestas,preguntas
SurveyID,,,
2014,1260,32760,26
2016,1433,88238,60
2017,756,57456,76
2018,417,31692,76
2019,352,26752,76


**Lo que he encontrado en el control de calidad:**
- Los datos están bien relacionados: todas las respuestas apuntan a preguntas y encuestas que existen, y cada persona aparece en una sola encuesta (4.218 personas distintas).
- El `-1` es un código que quiere decir "sin respuesta" o "no aplica". Sale, por ejemplo, en las 504 personas autónomas, que no responden las preguntas sobre su empresa. Lo voy a convertir en nulo.
- Hay 25 respuestas vacías, que también pasarán a nulo.
- Las filas repetidas (misma persona y misma pregunta) no son errores, son preguntas en las que se pueden marcar varias opciones (diagnósticos y puesto). Por eso la tabla de respuestas va a necesitar su propio identificador.

In [5]:
multiples = answer[answer.duplicated(["UserID", "QuestionID"], keep=False)]
(multiples.merge(question, left_on="QuestionID", right_on="questionid")
          .groupby(["QuestionID", "questiontext"]).size().rename("filas").reset_index())

,QuestionID,questiontext,filas
0,115,"If yes, what condition(s) have you been diagnosed with?",1006
1,116,"If maybe, what condition(s) do you believe you have?",567
2,117,Which of the following best describes your work position?,1875


### 2.1 Problemas en las preguntas de perfil

Las preguntas de edad, género y país necesitan una limpieza especial.

In [6]:
edad = pd.to_numeric(answer.loc[answer["QuestionID"] == 1, "AnswerText"], errors="coerce")
print("Edades fuera del rango 18-75:", sorted(edad[(edad < 18) | (edad > 75)].astype(int).tolist()))

genero = answer.loc[answer["QuestionID"] == 2, "AnswerText"]
print(f"\nGénero: {genero.nunique()} formas distintas de responder. Las 15 más frecuentes:")
print(genero.value_counts().head(15).to_dict())

pais = answer.loc[answer["QuestionID"] == 3, "AnswerText"]
print(f"\nPaís: {pais.nunique()} valores. Ejemplo de duplicado: ",
      pais[pais.str.contains("United States")].value_counts().to_dict())

Edades fuera del rango 18-75: [-29, -1, -1, -1, -1, -1, 0, 3, 5, 8, 11, 15, 17, 99, 323, 329]

Género: 101 formas distintas de responder. Las 15 más frecuentes:
{'Male': 2830, 'Female': 914, 'male': 212, 'female': 110, '-1': 24, 'Nonbinary': 8, 'non-binary': 6, 'Genderqueer': 4, 'Agender': 4, 'Genderfluid': 3, 'Non-binary': 3, 'None': 3, 'Male-ish': 2, 'Enby': 2, 'Trans woman': 2}

País: 81 valores. Ejemplo de duplicado:  {'United States of America': 1853, 'United States': 751}


**Lo que decido:**
- **Edad:** los valores imposibles (negativos, de 0 a 17 años, 99, 323, 329) los paso a nulo. Son 16 personas.
- **Género:** es una pregunta de texto libre con más de 100 formas distintas de escribirlo ("Male", "male", "M", "Cis Male", "non-binary", "nonbinary"…). Con expresiones regulares lo agrupo en cuatro categorías: *Hombre*, *Mujer*, *No binario / otra identidad* y *No responde / no clasificable*. Respeto lo que cada persona ha declarado: una mujer trans la clasifico como *Mujer*.
- **País:** junto las formas distintas de escribir el mismo país ("United States" y "United States of America") y le pongo el código ISO 3166-1 alfa-3 (USA, GBR, ESP…), como se recomienda en las buenas prácticas de visualización del curso.

In [7]:
PATRONES_GENERO = [
    ("No binario / otra identidad", r"non[- ]?binary|nonbinary|enby|\bnb\b|genderqueer|queer|agender|androgyn|fluid|flux|bigender"
                                    r"|neuter|demi(?!guy)|questioning|non-conforming|\bthey\b|both|multi|^other|^transgender$|^all$"),
    ("Mujer", r"trans[\s-]*(woman|female|feminine)|mtf|m2f|female \(trans\)|transgender woman"),
    ("Hombre", r"trans[\s-]*man|ftm|^male"),
    ("Mujer", r"female|woman|femmina|^f$|afab"),
    ("Hombre", r"male|\bman\b|^m$|guy|masculin|penis|\bswm\b|demiguy"),
]


def clasificar_genero(texto):
    """Agrupa la respuesta libre de género en cuatro categorías mediante expresiones regulares."""
    t = str(texto).strip().lower()
    for categoria, patron in PATRONES_GENERO:
        if re.search(patron, t):
            return categoria
    return "No responde / no clasificable"


ejemplos = ["Male", "female", "Cishet male", "male 9:1 female", "non-binary", "Trans woman", "male (trans, ftm)", "-1", "I am a wookie"]
pd.DataFrame({"respuesta": ejemplos, "categoria": [clasificar_genero(e) for e in ejemplos]})

,respuesta,categoria
0,Male,Hombre
1,female,Mujer
2,Cishet male,Hombre
3,male 9:1 female,Hombre
4,non-binary,No binario / otra identidad
5,Trans woman,Mujer
6,"male (trans, ftm)",Hombre
7,-1,No responde / no clasificable
8,I am a wookie,No responde / no clasificable


In [8]:
# País -> (código ISO 3166-1 alfa-3, nombre en español, región)
PAISES = {
    "United States of America": ("USA", "Estados Unidos", "América del Norte"), "United States": ("USA", "Estados Unidos", "América del Norte"),
    "Canada": ("CAN", "Canadá", "América del Norte"), "Mexico": ("MEX", "México", "Latinoamérica"),
    "Brazil": ("BRA", "Brasil", "Latinoamérica"), "Argentina": ("ARG", "Argentina", "Latinoamérica"), "Chile": ("CHL", "Chile", "Latinoamérica"),
    "Colombia": ("COL", "Colombia", "Latinoamérica"), "Costa Rica": ("CRI", "Costa Rica", "Latinoamérica"), "Uruguay": ("URY", "Uruguay", "Latinoamérica"),
    "Venezuela": ("VEN", "Venezuela", "Latinoamérica"), "Ecuador": ("ECU", "Ecuador", "Latinoamérica"), "Guatemala": ("GTM", "Guatemala", "Latinoamérica"),
    "Bahamas, The": ("BHS", "Bahamas", "Latinoamérica"),
    "United Kingdom": ("GBR", "Reino Unido", "Europa"), "Germany": ("DEU", "Alemania", "Europa"), "Netherlands": ("NLD", "Países Bajos", "Europa"),
    "France": ("FRA", "Francia", "Europa"), "Ireland": ("IRL", "Irlanda", "Europa"), "Sweden": ("SWE", "Suecia", "Europa"),
    "Switzerland": ("CHE", "Suiza", "Europa"), "Spain": ("ESP", "España", "Europa"), "Portugal": ("PRT", "Portugal", "Europa"),
    "Poland": ("POL", "Polonia", "Europa"), "Italy": ("ITA", "Italia", "Europa"), "Belgium": ("BEL", "Bélgica", "Europa"),
    "Russia": ("RUS", "Rusia", "Europa"), "Bulgaria": ("BGR", "Bulgaria", "Europa"), "Finland": ("FIN", "Finlandia", "Europa"),
    "Norway": ("NOR", "Noruega", "Europa"), "Austria": ("AUT", "Austria", "Europa"), "Romania": ("ROU", "Rumanía", "Europa"),
    "Denmark": ("DNK", "Dinamarca", "Europa"), "Greece": ("GRC", "Grecia", "Europa"), "Czech Republic": ("CZE", "Chequia", "Europa"),
    "Estonia": ("EST", "Estonia", "Europa"), "Hungary": ("HUN", "Hungría", "Europa"), "Croatia": ("HRV", "Croacia", "Europa"),
    "Serbia": ("SRB", "Serbia", "Europa"), "Bosnia and Herzegovina": ("BIH", "Bosnia y Herzegovina", "Europa"), "Ukraine": ("UKR", "Ucrania", "Europa"),
    "Latvia": ("LVA", "Letonia", "Europa"), "Lithuania": ("LTU", "Lituania", "Europa"), "Slovakia": ("SVK", "Eslovaquia", "Europa"),
    "Iceland": ("ISL", "Islandia", "Europa"), "Slovenia": ("SVN", "Eslovenia", "Europa"), "Moldova": ("MDA", "Moldavia", "Europa"),
    "Belarus": ("BLR", "Bielorrusia", "Europa"), "Macedonia": ("MKD", "Macedonia del Norte", "Europa"), "Georgia": ("GEO", "Georgia", "Europa"),
    "Australia": ("AUS", "Australia", "Oceanía"), "New Zealand": ("NZL", "Nueva Zelanda", "Oceanía"),
    "India": ("IND", "India", "Asia"), "Pakistan": ("PAK", "Pakistán", "Asia"), "Japan": ("JPN", "Japón", "Asia"),
    "Israel": ("ISR", "Israel", "Asia"), "Singapore": ("SGP", "Singapur", "Asia"), "Turkey": ("TUR", "Turquía", "Asia"),
    "Bangladesh": ("BGD", "Bangladés", "Asia"), "Afghanistan": ("AFG", "Afganistán", "Asia"), "Indonesia": ("IDN", "Indonesia", "Asia"),
    "China": ("CHN", "China", "Asia"), "Hong Kong": ("HKG", "Hong Kong", "Asia"), "Thailand": ("THA", "Tailandia", "Asia"),
    "Philippines": ("PHL", "Filipinas", "Asia"), "Vietnam": ("VNM", "Vietnam", "Asia"), "Brunei": ("BRN", "Brunéi", "Asia"),
    "Iran": ("IRN", "Irán", "Asia"), "Taiwan": ("TWN", "Taiwán", "Asia"), "Jordan": ("JOR", "Jordania", "Asia"),
    "Saudi Arabia": ("SAU", "Arabia Saudí", "Asia"),
    "South Africa": ("ZAF", "Sudáfrica", "África"), "Nigeria": ("NGA", "Nigeria", "África"), "Algeria": ("DZA", "Argelia", "África"),
    "Zimbabwe": ("ZWE", "Zimbabue", "África"), "Mauritius": ("MUS", "Mauricio", "África"), "Kenya": ("KEN", "Kenia", "África"),
    "Ethiopia": ("ETH", "Etiopía", "África"), "Ghana": ("GHA", "Ghana", "África"),
    "Botswana": ("BWA", "Botsuana", "África"), "Eritrea": ("ERI", "Eritrea", "África"), "Swaziland": ("SWZ", "Esuatini", "África"),
    "Luxembourg": ("LUX", "Luxemburgo", "Europa"), "United Arab Emirates": ("ARE", "Emiratos Árabes Unidos", "Asia"),
}
paises_respondidos = answer.loc[(answer["QuestionID"] == 3) | (answer["QuestionID"] == 50), "AnswerText"].unique()
sin_mapear = set(paises_respondidos) - set(PAISES) - {"Other", "-1"}
print("Países (de residencia o de trabajo) sin código ISO:", sin_mapear or "ninguno ('Other' y '-1' pasan a nulo)")

Países (de residencia o de trabajo) sin código ISO: ninguno ('Other' y '-1' pasan a nulo)


### 2.2 Traducción de preguntas y respuestas

Las preguntas y las respuestas están en inglés, y además la misma respuesta aparece escrita de formas distintas según el año (por ejemplo, *"Don't know"* en 2014 e *"I don't know"* desde 2016). Como el informe y el dashboard van en español, he hecho a mano dos tablas de traducción, que están en `01_datos/auxiliares/` para que se puedan revisar:

- `traduccion_preguntas.csv`: las 105 preguntas, en inglés y en español.
- `traduccion_respuestas.csv`: las respuestas cerradas (110 valores). Además de traducir, juntan las formas distintas de escribir la misma respuesta.

En las preguntas de perfil, la respuesta limpia utiliza los valores que ya he limpiado: la edad válida, el género en sus 4 categorías y el país en español. Dejo en su idioma original los estados de EE. UU. (son nombres propios) y los textos libres que escribió cada persona. La respuesta original la guardo siempre en la columna `respuesta_original`.

In [9]:
traduccion_preguntas = pd.read_csv(RUTA_AUXILIARES / "traduccion_preguntas.csv")
traduccion_respuestas = pd.read_csv(RUTA_AUXILIARES / "traduccion_respuestas.csv")
TRADUCCIONES = dict(zip(traduccion_respuestas["respuesta_en"], traduccion_respuestas["respuesta_es"]))
assert set(traduccion_preguntas["pregunta_id"]) == set(question["questionid"]), "Faltan preguntas por traducir"
print(f"{len(traduccion_preguntas)} preguntas y {len(TRADUCCIONES)} respuestas traducidas")
traduccion_preguntas.head(3)

105 preguntas y 110 respuestas traducidas


,pregunta_id,pregunta_en,pregunta_es
0,1,What is your age?,¿Qué edad tienes?
1,2,What is your gender?,¿Cuál es tu género?
2,3,What country do you live in?,¿En qué país vives?


In [10]:
PREGUNTAS_BINARIAS = {5, 7, 9}          # preguntas sí/no codificadas como 1/0 en el original
PREGUNTAS_PAIS = {3, 50}                # país de residencia y país de trabajo


def limpiar_respuesta(texto, pregunta_id):
    """Devuelve la respuesta normalizada y en español (nulo si es -1, vacía o imposible)."""
    t = str(texto).strip()
    if t in ("-1", ""):
        return None
    if pregunta_id == 1:
        edad = pd.to_numeric(t, errors="coerce")
        return t if 18 <= edad <= 75 else None
    if pregunta_id == 2:
        return clasificar_genero(t)
    if pregunta_id in PREGUNTAS_PAIS:
        return PAISES[t][1] if t in PAISES else None
    if pregunta_id in PREGUNTAS_BINARIAS:
        return {"1": "Sí", "0": "No"}.get(t, t)
    return TRADUCCIONES.get(t, t)


ejemplos = pd.DataFrame({"original": ["Don't know", "I don't know", "-1", "1", "Yes, I observed", "United States", "male", "329"],
                         "pregunta_id": [10, 10, 10, 7, 56, 3, 2, 1]})
ejemplos.assign(limpia=[limpiar_respuesta(t, q) for t, q in zip(ejemplos["original"], ejemplos["pregunta_id"])])

,original,pregunta_id,limpia
0,Don't know,10,No lo sé
1,I don't know,10,No lo sé
2,-1,10,NaN
3,1,7,Sí
4,"Yes, I observed",56,"Sí, lo observé"
5,United States,3,Estados Unidos
6,male,2,Hombre
7,329,1,NaN


## 3. Construcción del modelo relacional

**Las 7 tablas del modelo:**

| Tabla | Clave primaria | Claves foráneas | Qué es |
|---|---|---|---|
| `encuestas` | `encuesta_id` | — | Cada encuesta anual |
| `temas` | `tema_id` | — | Los temas en los que agrupo las preguntas (nueva) |
| `preguntas` | `pregunta_id` | `tema_id` | Cada pregunta, en español y en inglés, con un código corto |
| `paises` | `pais_iso3` | — | Los países con su código ISO y su región (nueva) |
| `tamanos_empresa` | `tamano_id` | — | Los tramos de tamaño de empresa, ordenados (nueva) |
| `personas` | `persona_id` | `encuesta_id`, `pais_iso3`, `tamano_id` | Una fila por persona, con su perfil limpio (nueva) |
| `respuestas` | `respuesta_id` | `persona_id`, `pregunta_id` | Cada respuesta |

**Normalización:** en la base original cada respuesta guardaba el año de la encuesta. Como cada persona responde una sola encuesta, paso el año a la tabla `personas` y lo quito de `respuestas`, porque el año depende de la persona y no de la respuesta (tercera forma normal). Cuando necesito el año desde una respuesta, hago una JOIN.

### 3.1 Temas y preguntas

In [11]:
TEMAS = {
    1: "Perfil personal y laboral",
    2: "Empresa actual: apoyo y recursos",
    3: "Empresas anteriores",
    4: "Hablar de salud mental y estigma",
    5: "Salud mental personal",
    6: "Impacto en el trabajo",
    7: "Experiencias y valoración del sector",
    8: "Texto libre y otros",
}
PREGUNTAS_POR_TEMA = {
    1: [1, 2, 3, 4, 5, 8, 9, 13, 22, 50, 51, 68, 89, 93, 117, 118],
    2: [10, 11, 14, 15, 16, 17, 64, 65, 94, 95, 96, 97],
    3: [23, 24, 25, 26, 27, 28, 69, 70, 72, 74, 76, 77, 108, 109, 110, 111, 112],
    4: [12, 18, 19, 29, 30, 31, 52, 53, 57, 58, 60, 62, 66, 67, 78, 79, 80, 81, 90, 91, 98, 99, 100, 101, 102,
        104, 105, 106, 107, 113, 114],
    5: [6, 7, 20, 21, 32, 33, 34, 115, 116],
    6: [48, 49, 54, 55, 92],
    7: [56, 83, 85],
    8: [59, 61, 63, 71, 73, 75, 82, 84, 86, 87, 88, 103],
}
tema_de_pregunta = {p: t for t, lista in PREGUNTAS_POR_TEMA.items() for p in lista}
assert set(tema_de_pregunta) == set(question["questionid"]), "Hay preguntas sin tema o temas con preguntas inexistentes"

# Código corto en español para las preguntas que se usan en el análisis
CODIGOS = {
    1: "edad", 2: "genero", 3: "pais_residencia", 5: "autonomo", 6: "historial_familiar", 7: "tratamiento_alguna_vez",
    8: "tamano_empresa", 9: "empresa_tech", 10: "prestaciones_salud_mental", 11: "anonimato_protegido",
    12: "lo_contaria_en_entrevista", 14: "conoce_opciones_cobertura", 15: "empresa_hablo_formalmente",
    16: "empresa_ofrece_recursos", 17: "facilidad_baja_medica", 18: "comodo_hablar_companeros",
    19: "comodo_hablar_supervisor", 33: "trastorno_actual", 34: "diagnosticado", 48: "interfiere_con_tratamiento",
    49: "interfiere_sin_tratamiento", 54: "productividad_afectada", 56: "vio_respuesta_negativa",
    65: "importancia_empresa_salud_mental", 83: "vio_respuesta_positiva", 85: "apoyo_sector_tech",
    92: "interfiere_trabajo_2014", 93: "teletrabajo_2014", 97: "facilidad_baja_medica_2014",
    98: "consecuencias_negativas_2014", 99: "dispuesto_hablar_companeros_2014", 100: "dispuesto_hablar_supervisor_2014",
    104: "consecuencias_negativas_2016", 113: "danaria_carrera", 115: "diagnostico", 117: "puesto", 118: "teletrabajo_2016",
}

temas = pd.DataFrame({"tema_id": list(TEMAS), "tema": list(TEMAS.values())})
preguntas = (traduccion_preguntas
             .assign(tema_id=lambda d: d["pregunta_id"].map(tema_de_pregunta),
                     codigo=lambda d: d["pregunta_id"].map(CODIGOS))
             [["pregunta_id", "tema_id", "codigo", "pregunta_es", "pregunta_en"]]
             .sort_values("pregunta_id").reset_index(drop=True))
print(f"{len(preguntas)} preguntas | {preguntas['codigo'].notna().sum()} con código de análisis")
preguntas.merge(temas, on="tema_id").groupby("tema").size().rename("preguntas").to_frame()

105 preguntas | 37 con código de análisis


,preguntas
tema,
Empresa actual: apoyo y recursos,12
Empresas anteriores,17
Experiencias y valoración del sector,3
Hablar de salud mental y estigma,31
Impacto en el trabajo,5
Perfil personal y laboral,16
Salud mental personal,9
Texto libre y otros,12


### 3.2 Encuestas, países y tamaños de empresa

In [12]:
encuestas = (survey.rename(columns={"SurveyID": "encuesta_id", "Description": "descripcion"})
             .assign(anio=lambda d: d["encuesta_id"],
                     descripcion=lambda d: "Encuesta OSMI de salud mental en tech " + d["anio"].astype(str))
             .merge(personas_por_anio["personas"].rename("num_personas"), left_on="encuesta_id", right_index=True)
             [["encuesta_id", "anio", "descripcion", "num_personas"]])

paises = (pd.DataFrame(sorted(set(PAISES.values())), columns=["pais_iso3", "pais", "region"])
          .drop_duplicates("pais_iso3").sort_values("pais_iso3").reset_index(drop=True))

TAMANOS = ["1-5", "6-25", "26-100", "100-500", "500-1000", "Más de 1000"]
tamanos_empresa = pd.DataFrame({"tamano_id": range(1, len(TAMANOS) + 1), "tramo_empleados": TAMANOS})

display(encuestas)
print(f"{len(paises)} países en {paises['region'].nunique()} regiones")
display(tamanos_empresa)

83 países en 6 regiones


,encuesta_id,anio,descripcion,num_personas
0,2014,2014,Encuesta OSMI de salud mental en tech 2014,1260
1,2016,2016,Encuesta OSMI de salud mental en tech 2016,1433
2,2017,2017,Encuesta OSMI de salud mental en tech 2017,756
3,2018,2018,Encuesta OSMI de salud mental en tech 2018,417
4,2019,2019,Encuesta OSMI de salud mental en tech 2019,352


,tamano_id,tramo_empleados
0,1,1-5
1,2,6-25
2,3,26-100
3,4,100-500
4,5,500-1000
5,6,Más de 1000


### 3.3 Personas

Creo una fila por persona pivotando sus respuestas de perfil (paso de una fila por respuesta a una columna por pregunta) y aplico la limpieza de la sección 2.

In [13]:
def pivotar_perfil(answer, ids_pregunta):
    """Pasa las respuestas de las preguntas indicadas a una fila por persona (una columna por pregunta)."""
    return (answer[answer["QuestionID"].isin(ids_pregunta)]
            .pivot_table(index=["UserID", "SurveyID"], columns="QuestionID", values="AnswerText", aggfunc="first")
            .reset_index())


perfil = pivotar_perfil(answer, [1, 2, 3, 5, 8, 9])
edad_num = pd.to_numeric(perfil[1], errors="coerce")
tamano_limpio = perfil[8].map(lambda t: limpiar_respuesta(t, 8))

personas = pd.DataFrame({
    "persona_id": perfil["UserID"],
    "encuesta_id": perfil["SurveyID"],
    "pais_iso3": perfil[3].map(lambda p: PAISES.get(p, (None,))[0]),
    "tamano_id": tamano_limpio.map({t: i for i, t in zip(tamanos_empresa["tamano_id"], TAMANOS)}).astype("Int64"),
    "edad": edad_num.where(edad_num.between(18, 75)).astype("Int64"),
    "genero": perfil[2].map(clasificar_genero),
    "autonomo": perfil[5].map({"1": 1, "0": 0}).astype("Int64"),
    "empresa_tech": perfil[9].map({"1": 1, "0": 0}).astype("Int64"),
}).sort_values("persona_id").reset_index(drop=True)

print(personas.shape)
display(personas["genero"].value_counts().to_frame("personas"))
personas.isna().sum().rename("nulos").to_frame()

(4218, 8)


,personas
genero,
Hombre,3059
Mujer,1042
No binario / otra identidad,74
No responde / no clasificable,43


,nulos
persona_id,0
encuesta_id,0
pais_iso3,4
tamano_id,504
edad,16
genero,0
autonomo,18
empresa_tech,504


**Los nulos que quedan en `personas` tienen explicación:**
- `tamano_id` y `empresa_tech`: son las 504 personas autónomas, que no tienen empresa, así que es correcto que estén vacíos.
- `edad`: las 16 edades imposibles.
- `pais_iso3`: 4 respuestas "Other" o en blanco.

### 3.4 Respuestas

In [14]:
respuestas = (answer.rename(columns={"UserID": "persona_id", "QuestionID": "pregunta_id", "AnswerText": "respuesta_original"})
              .assign(respuesta=lambda d: [limpiar_respuesta(t, q) for t, q in zip(d["respuesta_original"], d["pregunta_id"])])
              .sort_values(["persona_id", "pregunta_id"]).reset_index(drop=True))
respuestas.insert(0, "respuesta_id", range(1, len(respuestas) + 1))
respuestas = respuestas[["respuesta_id", "persona_id", "pregunta_id", "respuesta_original", "respuesta"]]

print(f"{len(respuestas):,} respuestas | nulas tras limpiar (-1 o vacías): {respuestas['respuesta'].isna().sum():,}")
respuestas.head()

236,898 respuestas | nulas tras limpiar (-1 o vacías): 56,639


,respuesta_id,persona_id,pregunta_id,respuesta_original,respuesta
0,1,1,1,37,37
1,2,1,2,Female,Mujer
2,3,1,3,United States,Estados Unidos
3,4,1,4,Illinois,Illinois
4,5,1,5,-1,NaN


## 4. Integridad referencial y resumen del modelo

In [15]:
def comprobar_integridad(hija, columna, padre, clave):
    """Devuelve cuántos valores no nulos de la clave foránea no existen en la tabla padre."""
    valores = hija[columna].dropna()
    return int((~valores.isin(padre[clave])).sum())


relaciones = [
    ("preguntas.tema_id -> temas", preguntas, "tema_id", temas, "tema_id"),
    ("personas.encuesta_id -> encuestas", personas, "encuesta_id", encuestas, "encuesta_id"),
    ("personas.pais_iso3 -> paises", personas, "pais_iso3", paises, "pais_iso3"),
    ("personas.tamano_id -> tamanos_empresa", personas, "tamano_id", tamanos_empresa, "tamano_id"),
    ("respuestas.persona_id -> personas", respuestas, "persona_id", personas, "persona_id"),
    ("respuestas.pregunta_id -> preguntas", respuestas, "pregunta_id", preguntas, "pregunta_id"),
]
integridad = pd.DataFrame([(n, comprobar_integridad(h, c, p, k)) for n, h, c, p, k in relaciones],
                          columns=["relacion", "valores_huerfanos"])
assert integridad["valores_huerfanos"].sum() == 0
integridad

,relacion,valores_huerfanos
0,preguntas.tema_id -> temas,0
1,personas.encuesta_id -> encuestas,0
2,personas.pais_iso3 -> paises,0
3,personas.tamano_id -> tamanos_empresa,0
4,respuestas.persona_id -> personas,0
5,respuestas.pregunta_id -> preguntas,0


In [16]:
TABLAS = {"encuestas": encuestas, "temas": temas, "preguntas": preguntas, "paises": paises,
          "tamanos_empresa": tamanos_empresa, "personas": personas, "respuestas": respuestas}
pd.DataFrame({"filas": {n: len(t) for n, t in TABLAS.items()}, "columnas": {n: t.shape[1] for n, t in TABLAS.items()}})

,filas,columnas
encuestas,5,4
temas,8,2
preguntas,105,5
paises,83,3
tamanos_empresa,6,2
personas,4218,8
respuestas,236898,5


## 5. Guardo los resultados

- **CSV** en `01_datos/procesados/`: son los datos que cargo en Power BI, y uno de los entregables del proyecto.
- **SQLite** (`salud_mental_tech.db`): la misma base, para consultarla desde los notebooks de Python.
- **Script de MySQL** (`03_sql/01_crear_base_datos.sql`): crea la base completa en MySQL. He seguido las buenas prácticas del curso: empiezo con `DROP DATABASE IF EXISTS`, creo las tablas con sus claves primarias y foráneas y meto los datos sin desactivar la integridad referencial. Para eso cargo las tablas en orden: primero las tablas padre y después las hijas.

In [17]:
for nombre, tabla in TABLAS.items():
    tabla.to_csv(RUTA_PROCESADOS / f"{nombre}.csv", index=False, encoding="utf-8")

RUTA_SQLITE.unlink(missing_ok=True)
with sqlite3.connect(RUTA_SQLITE) as con:
    for nombre, tabla in TABLAS.items():
        tabla.to_sql(nombre, con, index=False)
print("CSV y SQLite guardados:", sorted(p.name for p in RUTA_PROCESADOS.iterdir()))

CSV y SQLite guardados: ['encuestas.csv', 'paises.csv', 'personas.csv', 'preguntas.csv', 'respuestas.csv', 'salud_mental_tech.db', 'tamanos_empresa.csv', 'temas.csv']


In [18]:
ESQUEMA_MYSQL = """
CREATE TABLE encuestas (
    encuesta_id   INT NOT NULL,
    anio          INT NOT NULL,
    descripcion   VARCHAR(100),
    num_personas  INT,
    PRIMARY KEY (encuesta_id)
);

CREATE TABLE temas (
    tema_id  INT NOT NULL,
    tema     VARCHAR(60) NOT NULL,
    PRIMARY KEY (tema_id)
);

CREATE TABLE preguntas (
    pregunta_id     INT NOT NULL,
    tema_id         INT NOT NULL,
    codigo          VARCHAR(50),
    pregunta_es     VARCHAR(400) NOT NULL,
    pregunta_en     VARCHAR(400) NOT NULL,
    PRIMARY KEY (pregunta_id),
    UNIQUE KEY uk_preguntas_codigo (codigo),
    CONSTRAINT fk_preguntas_temas FOREIGN KEY (tema_id) REFERENCES temas (tema_id)
);

CREATE TABLE paises (
    pais_iso3  CHAR(3) NOT NULL,
    pais       VARCHAR(40) NOT NULL,
    region     VARCHAR(30) NOT NULL,
    PRIMARY KEY (pais_iso3)
);

CREATE TABLE tamanos_empresa (
    tamano_id        INT NOT NULL,
    tramo_empleados  VARCHAR(20) NOT NULL,
    PRIMARY KEY (tamano_id)
);

CREATE TABLE personas (
    persona_id    INT NOT NULL,
    encuesta_id   INT NOT NULL,
    pais_iso3     CHAR(3),
    tamano_id     INT,
    edad          INT,
    genero        VARCHAR(40),
    autonomo      TINYINT,
    empresa_tech  TINYINT,
    PRIMARY KEY (persona_id),
    CONSTRAINT fk_personas_encuestas FOREIGN KEY (encuesta_id) REFERENCES encuestas (encuesta_id),
    CONSTRAINT fk_personas_paises FOREIGN KEY (pais_iso3) REFERENCES paises (pais_iso3),
    CONSTRAINT fk_personas_tamanos FOREIGN KEY (tamano_id) REFERENCES tamanos_empresa (tamano_id)
);

CREATE TABLE respuestas (
    respuesta_id        INT NOT NULL,
    persona_id          INT NOT NULL,
    pregunta_id         INT NOT NULL,
    respuesta_original  TEXT,
    respuesta           TEXT,
    PRIMARY KEY (respuesta_id),
    CONSTRAINT fk_respuestas_personas FOREIGN KEY (persona_id) REFERENCES personas (persona_id),
    CONSTRAINT fk_respuestas_preguntas FOREIGN KEY (pregunta_id) REFERENCES preguntas (pregunta_id)
);
"""


def valor_sql(valor):
    """Convierte un valor de Python en un literal SQL seguro (NULL, número o texto escapado)."""
    if valor is None or valor is pd.NA or (isinstance(valor, float) and np.isnan(valor)):
        return "NULL"
    if isinstance(valor, (int, np.integer)):
        return str(int(valor))
    texto = str(valor).replace("\\", "\\\\").replace("'", "''")
    return f"'{texto}'"


def escribir_inserts(fichero, nombre, tabla, tamano_lote=2000):
    """Escribe la tabla como sentencias INSERT por lotes."""
    columnas = ", ".join(tabla.columns)
    filas = list(tabla.astype(object).where(tabla.notna(), None).itertuples(index=False, name=None))
    for inicio in range(0, len(filas), tamano_lote):
        valores = ",\n".join("(" + ", ".join(valor_sql(v) for v in fila) + ")" for fila in filas[inicio:inicio + tamano_lote])
        fichero.write(f"INSERT INTO {nombre} ({columnas}) VALUES\n{valores};\n")


with open(RUTA_MYSQL, "w", encoding="utf-8", newline="\n") as f:
    f.write("-- ====================================================================\n"
            "-- 01 · Crear la base de datos salud_mental_tech (MySQL 8)\n"
            "-- Proyecto final · Hugo Nascimento · IT Academy\n"
            "-- Generado por 02_notebooks/01_limpieza_y_modelo.ipynb\n"
            "-- Uso: MySQL Workbench > Server > Data Import > Import from Self-Contained File\n"
            "-- ====================================================================\n\n"
            "DROP DATABASE IF EXISTS salud_mental_tech;\n"
            "CREATE DATABASE salud_mental_tech CHARACTER SET utf8mb4;\n"
            "USE salud_mental_tech;\n"
            "SET NAMES utf8mb4;\n")
    f.write(ESQUEMA_MYSQL + "\n")
    f.write("-- Carga de datos: primero las tablas padre y después las hijas (la integridad referencial se mantiene activa)\n")
    for nombre, tabla in TABLAS.items():
        escribir_inserts(f, nombre, tabla)
    f.write("\n-- Comprobación: número de filas por tabla\n")
    f.write("\nUNION ALL\n".join(f"SELECT '{n}' AS tabla, COUNT(*) AS filas FROM {n}" for n in TABLAS) + ";\n")

print(f"Script MySQL: {RUTA_MYSQL.stat().st_size / 1e6:.1f} MB")

Script MySQL: 10.7 MB


## 6. Conclusiones

- La base de datos original estaba bien relacionada (no hay respuestas sueltas), pero no tenía claves y las respuestas estaban escritas de muchas formas distintas.
- He documentado toda la limpieza: el `-1` y los vacíos pasan a nulo, las edades imposibles pasan a nulo, el género pasa de texto libre a 4 categorías, los países llevan el código ISO 3166-1 alfa-3 y el nombre en español, y las preguntas y respuestas cerradas están traducidas al español con tablas que se pueden revisar.
- El resultado es un modelo relacional de 7 tablas, con claves primarias y foráneas y normalizado (el año depende de la persona, no de la respuesta).
- Lo siguiente: las consultas de `03_sql/02_consultas.sql` y el análisis exploratorio del notebook `02_analisis_exploratorio.ipynb`.